# R13 — Sintonia e IA: do registro de telemetria aos coeficientes no chip

Microcontroladores — DENE/UFMT. Notebook do roteiro R13 (aula 13).

O fluxo tem quatro etapas, e cada uma usa um arquivo que **o seu grupo** gravou:

| Etapa | Quando | Entrada | Saída |
|---|---|---|---|
| A. Identificar a planta | antes da sessão | `degrau.csv` (R10) | ganho $G$, constante de tempo $\tau$, atraso $\theta$ |
| B. Sintonizar o PI no simulador | antes da sessão | o modelo da etapa A | `KP_Q8`, `KI_Q8` |
| C. Rotular e treinar | na sessão, depois da Tarefa 3 | `pi.csv` (telemetria do PI) | uma reta e uma rede 2-3-1 |
| D. Quantizar e exportar | na sessão | os modelos da etapa C | `modelo.h` |

**Regra do curso:** o treino roda aqui; o microcontrolador só executa o que foi aprendido, em inteiros.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# No Colab: escolha degrau.csv (e, na etapa C, pi.csv) na janela que abrir.
try:
    from google.colab import files
    enviados = files.upload()
except ImportError:
    print("Fora do Colab: os arquivos devem estar na pasta deste notebook.")

## A. Identificar a planta a partir do degrau

O `degrau.csv` do R10 tem as colunas `t_ms,temp_d,alvo_d,aquecedor,comutacoes`. Usamos só o trecho com o aquecedor ligado.

O modelo é de primeira ordem com atraso: com a saída $u$ entre 0 e 1,

$$\frac{dT}{dt} = \frac{T_{amb} + G\,u - T}{\tau}, \qquad \text{e a resposta ao degrau é } T(t) = T_0 + G\,(1 - e^{-(t-\theta)/\tau}).$$

In [ ]:
d = pd.read_csv("degrau.csv")
d = d[d.iloc[:, 3] == 1]                       # 4a coluna: aquecedor ligado
t = (d.iloc[:, 0] - d.iloc[0, 0]).to_numpy() / 1000.0     # s
T = d.iloc[:, 1].to_numpy() / 10.0                        # graus
T0 = T[:10].mean()
TAMB = T0

def degrau(t, G, tau, theta):
    s = np.clip(t - theta, 0, None)
    return T0 + G * (1 - np.exp(-s / tau))

(G, tau, theta), cov = curve_fit(degrau, t, T, p0=[30, 1800, 30],
                                 bounds=([1, 60, 0], [200, 30000, 600]))
eG, etau, eth = np.sqrt(np.diag(cov))
print(f"T0 = {T0:.1f} C")
print(f"G     = {G:6.1f} +- {eG:.1f} C    (quanto a planta sobe com 100 %)")
print(f"tau   = {tau/60:6.1f} +- {etau/60:.1f} min")
print(f"theta = {theta:6.0f} +- {eth:.0f} s")
print(f"taxa inicial a = G/tau = {G/tau*60:.2f} C/min")

plt.figure(figsize=(8, 3.5))
plt.plot(t/60, T, ".", ms=2, label="medido")
plt.plot(t/60, degrau(t, G, tau, theta), label="modelo")
plt.xlabel("min"); plt.ylabel("C"); plt.legend(); plt.grid(alpha=.3); plt.show()

**Pergunta (Tarefa 1):** as incertezas de $G$ e de $\tau$ são pequenas? Com dez minutos de degrau e uma constante de tempo de dezenas de minutos, a curva quase não achata: o ajuste mede bem a **taxa inicial** $G/\tau$, e mal cada um separadamente. A etapa C vai medir $G$ de outro jeito.

## B. Sintonizar o PI no simulador — com a quantização

O simulador repete **exatamente** a conta do firmware: erro em décimos, ganhos em Q8, saída de 0 a 1023, e a temperatura medida arredondada para o degrau de 0,488 C do conversor (aula 13: *o simulador mente se não tiver a quantização*).

In [ ]:
TS = 1.0                         # s: o controle roda uma vez por segundo
LSB = 5000 / 1024 / 10           # graus por codigo do conversor

def simular(KP_Q8, KI_Q8, alvo=40.0, minutos=60, antiwindup=True, ff=None,
            T_ini=None, alvo2=None, t_troca_min=None, G_=None, tau_=None):
    G_ = G if G_ is None else G_
    tau_ = tau if tau_ is None else tau_
    n = int(minutos * 60 / TS)
    fila = [0.0] * (int(round(theta / TS)) + 1)     # o atraso da planta
    Tp = TAMB if T_ini is None else T_ini
    integral = 0
    Tm_s, u_s = [], []
    for k in range(n):
        a = alvo if (alvo2 is None or k * TS < t_troca_min * 60) else alvo2
        Tm = np.floor(Tp / LSB) * LSB                 # o que o conversor ve
        e = int(round((a - Tm) * 10))                 # decimos
        u_ff = 0 if ff is None else ff(a, Tm)
        u = u_ff + ((KP_Q8 * e + KI_Q8 * integral) >> 8)   # igual ao firmware
        if u > 1023:
            u = 1023
        elif u < 0:
            u = 0
        elif antiwindup:
            integral += e                             # so integra fora da saturacao
        if not antiwindup:
            integral += e
        fila.append(u / 1023)
        Tp += TS / tau_ * (TAMB + G_ * fila.pop(0) - Tp)
        Tm_s.append(Tm); u_s.append(u)
    return np.arange(n) * TS / 60, np.array(Tm_s), np.array(u_s)

In [ ]:
ALVO = 40.0
linhas = []
for kp in [256, 512, 1024, 2048, 4096, 8192]:
    for ki in [0, 1, 2, 4, 8, 16]:
        tm, Tm, u = simular(kp, ki, alvo=ALVO, minutos=90)
        depois = tm > 30
        linhas.append(dict(KP_Q8=kp, KI_Q8=ki,
                           erro_medio=np.mean(np.abs(Tm[depois] - ALVO)),
                           sobressinal=max(0.0, Tm.max() - ALVO)))
tabela = pd.DataFrame(linhas)
tabela["custo"] = tabela.erro_medio + 0.5 * tabela.sobressinal
print(tabela.sort_values("custo").head(8).to_string(index=False))

In [ ]:
# Escolha (ou troque por outra linha da tabela):
KP_Q8, KI_Q8 = [int(v) for v in tabela.sort_values("custo").iloc[0][["KP_Q8", "KI_Q8"]]]
print("KP_Q8 =", KP_Q8, "  KI_Q8 =", KI_Q8)

fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
for nome, kw in [("P sozinho", dict(KI_Q8=0)),
                 ("PI sem protecao", dict(antiwindup=False)),
                 ("PI com protecao", dict())]:
    tm, Tm, u = simular(KP_Q8, kw.get("KI_Q8", KI_Q8), alvo=ALVO, minutos=90,
                        antiwindup=kw.get("antiwindup", True))
    ax[0].plot(tm, Tm, label=nome); ax[1].plot(tm, u)
    print(f"{nome:16s}: final {Tm[-600:].mean():5.2f} C, sobressinal {max(0, Tm.max()-ALVO):4.2f} C")
ax[0].axhline(ALVO, color="k", lw=.8, ls="--"); ax[0].legend(); ax[0].set_ylabel("C")
ax[1].set_ylabel("u (0-1023)"); ax[1].set_xlabel("min"); plt.show()

**Perguntas (Tarefa 2):** onde o P sozinho estabiliza, e por quê (P2)? Quanto de sobressinal o PI sem proteção produz (P3)? Anote `KP_Q8` e `KI_Q8`: eles vão para o firmware da Tarefa 3.

## C. Rotular os dados da bancada e treinar

O `pi.csv` é a telemetria da Tarefa 3: `t_ms,temp_d,alvo_d,u,ff`. A rede aprende a **saída de regime** — a que mantém a planta parada — e por isso só entram as amostras com a temperatura estável (aula 13: *rotular não é gravar*).

In [ ]:
try:
    enviados = files.upload()          # escolha pi.csv
except NameError:
    pass
p = pd.read_csv("pi.csv")
p.columns = ["t_ms", "temp_d", "alvo_d", "u", "ff"]
p["t_min"] = (p.t_ms - p.t_ms.iloc[0]) / 60000
p["T"] = p.temp_d / 10
p["dTdt"] = p["T"].rolling(120, center=True).mean().diff(60)     # graus por minuto
regime = p[(p.dTdt.abs() < 0.02) & ((p.alvo_d - p.temp_d).abs() <= 5)].dropna()
print(f"{len(p)} amostras, {len(regime)} em regime")

plt.figure(figsize=(8, 3.5))
plt.plot(p.t_min, p["T"], lw=.8, label="temperatura")
plt.plot(p.t_min, p.alvo_d / 10, "k--", lw=.8, label="alvo")
plt.plot(regime.t_min, regime["T"], ".", ms=3, label="regime (rotulado)")
plt.xlabel("min"); plt.ylabel("C"); plt.legend(); plt.grid(alpha=.3); plt.show()

In [ ]:
from sklearn.neural_network import MLPRegressor

Y = (regime.u / 1023).to_numpy()                                # saida de regime, 0..1
X = np.column_stack([(regime["T"] - 25) / 25,                   # x1 = (T - 25)/25
                     (regime.alvo_d - regime.temp_d) / 100])    # x2 = e/10, e em graus

# Reta: dois parametros, Tamb (medido no degrau) e a inclinacao c1.
# A fisica diz que a saida de regime e (T - Tamb)/G: a reta passa pelo ambiente.
x = (regime["T"] - TAMB).to_numpy()
c1 = float(np.sum(x * Y) / np.sum(x * x))
c0 = 0.0
erro_reta = np.sqrt(np.mean((c1 * x - Y) ** 2))

# Rede 2-3-1 com retificacao: treze parametros.
rede = MLPRegressor(hidden_layer_sizes=(3,), activation="relu", max_iter=20000,
                    random_state=0).fit(X, Y)
erro_rede = np.sqrt(np.mean((rede.predict(X) - Y) ** 2))

print(f"reta: u = {c1:.4f} (T - Tamb)                erro {erro_reta*100:.2f} %")
print(f"rede: 13 parametros                          erro {erro_rede*100:.2f} %")
G_c = 1 / c1                      # o ganho da planta, medido pelo regime
tau_c = G_c / (G / tau)           # a taxa inicial G/tau da etapa A e confiavel
print(f"ganho da planta pelo regime: G = 1/c1 = {G_c:.1f} C  (etapa A: {G:.1f} +- {eG:.1f})")
print(f"modelo corrigido: G = {G_c:.1f} C, tau = {tau_c/60:.1f} min")

**Perguntas (Tarefa 4):** quem erra menos, a reta ou a rede? E $1/c_1$ — o coeficiente da reta — é o ganho $G$ da planta: na primeira ordem, a saída de regime é $(T - T_{amb})/G$. Compare com o $G$ da etapa A, que dez minutos de degrau mediam mal. O modelo corrigido é o que a etapa D usa.

## D. Quantizar em Q8, verificar, resimular e exportar

Os pesos viram inteiros `int16` em Q8 (valor real = inteiro / 256). Antes de gravar: o maior peso cabe em ±128? E o controle **com os pesos arredondados** continua bom? (aula 13: *um modelo validado em ponto flutuante não é o modelo que vai rodar*.)

In [ ]:
q8 = lambda v: np.round(np.asarray(v) * 256).astype(int)
W1, b1 = q8(rede.coefs_[0].T), q8(rede.intercepts_[0])        # 3x2 e 3
W2, b2 = q8(rede.coefs_[1][:, 0]), int(q8(rede.intercepts_[1])[0])
maior = max(np.abs(rede.coefs_[0]).max(), np.abs(rede.coefs_[1]).max(),
            np.abs(rede.intercepts_[0]).max(), abs(rede.intercepts_[1][0]))
print(f"maior peso em modulo: {maior:.2f}  ->  {'cabe' if maior < 128 else 'NAO CABE'} em Q8")

FF_B = int(round(c0 * 1023))                  # reta no firmware:
FF_A_Q8 = int(round(c1 * 1023 / 10 * 256))    # u_ff = B + (A_Q8 * (alvo_d - TAMB_D)) >> 8
TAMB_D = int(round(TAMB * 10))

def satura(v): return max(-32768, min(32767, v))
def cdiv(a, b): return int(a / b)             # divisao inteira do C: trunca para zero

def ff_rede(alvo, Tm):                        # a mesma conta do firmware (aula 13)
    t_d, a_d = int(round(Tm * 10)), int(round(alvo * 10))
    x = [cdiv((t_d - 250) * 256, 250), cdiv((a_d - t_d) * 256, 100)]
    h = []
    for j in range(3):
        acc = (int(b1[j]) << 8) + sum(int(W1[j][i]) * x[i] for i in range(2))
        acc >>= 8
        h.append(satura(acc) if acc > 0 else 0)
    s = (b2 << 8) + sum(int(W2[j]) * h[j] for j in range(3))
    return (satura(s >> 8) * 1023) >> 8

def ff_reta(alvo, Tm):
    return FF_B + ((FF_A_Q8 * (int(round(alvo * 10)) - TAMB_D)) >> 8)

fig, ax = plt.subplots(figsize=(8, 3.5))
for nome, ff in [("PI", None), ("PI + reta", ff_reta), ("PI + rede (Q8)", ff_rede)]:
    tm, Tm, u = simular(KP_Q8, KI_Q8, alvo=40, alvo2=45, t_troca_min=60,
                        minutos=120, ff=ff, T_ini=40.0, G_=G_c, tau_=tau_c)
    ax.plot(tm, Tm, label=nome)
ax.axhline(45, color="k", lw=.8, ls="--"); ax.set_xlabel("min"); ax.set_ylabel("C")
ax.legend(); plt.show()

In [ ]:
FF_TIPO = 1          # 1 = reta, 2 = rede: escolha a que a Tarefa 4 mostrou melhor
h = f'''/* modelo.h -- gerado pelo notebook R13. Nao edite a mao. */
#ifndef MODELO_H
#define MODELO_H
#include <stdint.h>

#define KP_Q8    {KP_Q8}
#define KI_Q8    {KI_Q8}
#define FF_TIPO  {FF_TIPO}          /* 0 nenhum, 1 reta, 2 rede */

/* reta: u_ff = FF_B + (FF_A_Q8 * (alvo_d - TAMB_D)) >> 8 */
#define FF_B     {FF_B}
#define FF_A_Q8  {FF_A_Q8}
#define TAMB_D   {TAMB_D}

/* rede 2-3-1, pesos em Q8 */
static const int16_t W1[3][2] = {{ {{{W1[0][0]}, {W1[0][1]}}}, {{{W1[1][0]}, {W1[1][1]}}}, {{{W1[2][0]}, {W1[2][1]}}} }};
static const int16_t b1[3]    = {{ {b1[0]}, {b1[1]}, {b1[2]} }};
static const int16_t W2[3]    = {{ {W2[0]}, {W2[1]}, {W2[2]} }};
static const int16_t b2       = {b2};

#endif
'''
open("modelo.h", "w").write(h)
print(h)
try:
    files.download("modelo.h")
except NameError:
    pass